In [1]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, average_precision_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, auc
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

PROJECT_ROOT = os.path.abspath(os.getcwd())
DATA_PATH = os.path.join(PROJECT_ROOT, "Crop_Dataset.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "Data", "artifacts")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH:", DATA_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)


PROJECT_ROOT: c:\Users\manur\OneDrive\Desktop\DSGP\Weather-Prediction-and-Crop-Recommendation-System-
DATA_PATH: c:\Users\manur\OneDrive\Desktop\DSGP\Weather-Prediction-and-Crop-Recommendation-System-\Crop_Dataset.csv
OUTPUT_DIR: c:\Users\manur\OneDrive\Desktop\DSGP\Weather-Prediction-and-Crop-Recommendation-System-\Data\artifacts


In [2]:
MODEL_PATH = os.path.join(OUTPUT_DIR, 'best_ranking_model.joblib')
META_PATH = os.path.join(OUTPUT_DIR, 'training_meta.json')

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")
if not os.path.exists(META_PATH):
    raise FileNotFoundError(f"Meta file not found: {META_PATH}")

model = joblib.load(MODEL_PATH)
with open(META_PATH, 'r', encoding='utf-8') as f:
    meta = json.load(f)

crop_list = meta.get('crop_list', [])
threshold_default = meta.get('threshold_default', 0.5)
print("Loaded model and artifacts.")
print(f"crop_list: {len(crop_list)} entries, default threshold: {threshold_default}")

def build_query_table(weather: dict, soil: dict) -> pd.DataFrame:
    sunshine = weather.get('sunshine_hours', 0.0)
    rows = []
    for crop in crop_list:
        rows.append({
            'crop': crop,
            'temperature': float(weather['temperature']),
            'rainfall': float(weather['rainfall']),
            'sunshine_hours': float(sunshine),
            'ph': float(soil['ph']),
            'organic_carbon': float(soil['organic_carbon']),
            'cec': float(soil['cec']),
            'awc': float(soil['awc']),
            'bulk_density': float(soil['bulk_density']),
            'texture_code': float(soil['texture_code']),
            'scenario_id': 0
        })
    return pd.DataFrame(rows)


def rank_top5_crops(weather: dict, soil: dict, top_k: int = 5, threshold: float = None):
    if threshold is None:
        threshold = threshold_default
    query_df = build_query_table(weather, soil)
    scores = model.predict_proba(query_df)[:, 1]
    result = pd.DataFrame({'crop': query_df['crop'], 'score': scores})
    result['class'] = result['score'].apply(lambda x: 'Suitable' if x >= threshold else 'Unsuitable')
    result = result.sort_values('score', ascending=False).reset_index(drop=True)
    return result.head(top_k)

print('Helper functions defined.')


Loaded model and artifacts.
crop_list: 21 entries, default threshold: 0.5
Helper functions defined.


In [3]:
weather_example = {
    "temperature": 28.0,
    "rainfall": 1200.0,
    "sunshine_hours": 6.5
}

soil_example = {
    "ph": 5.2,
    "organic_carbon": 1.8,
    "cec": 12.5,
    "awc": 0.018,
    "bulk_density": 1.25,
    "texture_code": 3
}

top5 = rank_top5_crops(weather_example, soil_example, top_k=5, threshold=0.5)
display(top5)

,crop,score,class
0,Papaya,0.997118,Suitable
1,Banana,0.996529,Suitable
2,Passion Fruit,0.995625,Suitable
3,Rambutan,0.995081,Suitable
4,Pineapple,0.992556,Suitable
